In [1]:
!pip install sentence-transformers

## Embedding Engine

In [2]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from typing import Dict, List, Tuple

class EmbeddingEngine:
    def __init__(self, model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2'):
        """
        Inicializa el motor de Deep Learning.
        Cargar el modelo en RAM es una operación costosa, por lo que 
        solo debe hacerse una vez al instanciar la clase (Patrón Singleton en producción).
        """
        print(f"⚙️ Cargando modelo Transformer: {model_name}...")
        self.modelo = SentenceTransformer(model_name)
        self.anclajes: Dict[str, np.ndarray] = {}
        print("✅ Modelo cargado en memoria.")

    def generar_vector(self, texto: str) -> np.ndarray:
        """
        La capa de codificación. Transforma texto plano en un tensor matemático.
        """
        # El parámetro convert_to_tensor=True es vital para acelerar cálculos con PyTorch
        vector = self.modelo.encode(texto, convert_to_tensor=True)
        return vector

    def crear_anclajes(self, categorias_json: Dict[str, str]):
        """
        Ingesta el JSON de las 4 Áreas del CENEVAL y crea los centros de gravedad.
        """
        print("🌌 Generando el espacio latente para las áreas de conocimiento...")
        for area, descripcion in categorias_json.items():
            vector_ancla = self.generar_vector(descripcion)
            self.anclajes[area] = vector_ancla
            print(f"  -> Anclaje creado para: {area} (Dimensiones: {len(vector_ancla)})")

    def clasificar_pregunta(self, pregunta: str) -> Tuple[str, float]:
        """
        El orquestador de inferencia. Toma una pregunta nueva, genera su vector
        y calcula la distancia geométrica contra todas las áreas conocidas.
        """
        if not self.anclajes:
            raise ValueError("⚠️ No hay anclajes definidos. Ejecuta crear_anclajes() primero.")

        vector_pregunta = self.generar_vector(pregunta)
        
        mejor_area = None
        mayor_similitud = -1.0 # La similitud del coseno va de -1 (opuestos) a 1 (idénticos)

        # Calculamos la distancia contra las 4 galaxias principales
        for area, vector_ancla in self.anclajes.items():
            # util.cos_sim devuelve una matriz, extraemos el valor escalar (.item())
            similitud = util.cos_sim(vector_pregunta, vector_ancla).item()
            
            if similitud > mayor_similitud:
                mayor_similitud = similitud
                mejor_area = area

        return mejor_area, mayor_similitud

## Prueba del modelo

In [3]:
# 1. Instanciamos el motor (descargará el modelo la primera vez)
motor_nlp = EmbeddingEngine()

# 2. Le pasamos el JSON con los 4 prompts que definimos anteriormente
categorias = {
    "Algoritmia": "Fundamentos lógicos y matemáticos...",
    "Desarrollo de software de base": "Interacción de bajo nivel...",
    "Desarrollo de software de aplicación": "Ciclo de vida para la creación...",
    "Soluciones de cómputo inteligente": "Sistemas avanzados y procesamiento..."
}
motor_nlp.crear_anclajes(categorias)

# 3. La prueba de fuego: Inferencia sin palabras clave exactas
pregunta_prueba = "¿Qué estructura de datos utiliza el principio LIFO (Last In, First Out)?"
area_predicha, score = motor_nlp.clasificar_pregunta(pregunta_prueba)

print(f"\n📝 Pregunta: {pregunta_prueba}")
print(f"🎯 Clasificación IA: {area_predicha} (Confianza: {score:.4f})")

⚙️ Cargando modelo Transformer: paraphrase-multilingual-MiniLM-L12-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo cargado en memoria.
🌌 Generando el espacio latente para las áreas de conocimiento...
  -> Anclaje creado para: Algoritmia (Dimensiones: 384)
  -> Anclaje creado para: Desarrollo de software de base (Dimensiones: 384)
  -> Anclaje creado para: Desarrollo de software de aplicación (Dimensiones: 384)
  -> Anclaje creado para: Soluciones de cómputo inteligente (Dimensiones: 384)

📝 Pregunta: ¿Qué estructura de datos utiliza el principio LIFO (Last In, First Out)?
🎯 Clasificación IA: Algoritmia (Confianza: 0.3860)
